In [ ]:
import json
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec
import pandas as pd
import os

VOLUME_PATH = "/Volumes/workspace/mlpab4886e6/mlpab4886e6_ft"

# Read metrics from volume
with open(os.path.join(VOLUME_PATH, "metrics.json")) as f:
    metrics = json.load(f)
print("Metrics:", metrics)

# Set MLflow tracking URI to Databricks
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

client = MlflowClient()

# Create experiment
experiment_path = "/Users/benedict@logicalclocks.com/mlpab4886e6/ftmodel0b3133_exp"
try:
    experiment_id = client.create_experiment(experiment_path)
except Exception:
    experiment_id = client.get_experiment_by_name(experiment_path).experiment_id

print(f"Experiment ID: {experiment_id}")

# Define model signature
input_schema = Schema([ColSpec("string", "text")])
output_schema = Schema([ColSpec("string", "result")])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

model_src = os.path.join(VOLUME_PATH, "finetuned_model.npz")
metrics_src = os.path.join(VOLUME_PATH, "metrics.json")

class NpzModel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        import numpy as np
        self.data = np.load(context.artifacts["model_file"])
    def predict(self, context, model_input):
        return pd.DataFrame({"result": ["ok"] * len(model_input)})

artifacts = {
    "model_file": model_src,
    "metrics_file": metrics_src
}

# Create a run and log model
with mlflow.start_run(experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    # Log metrics
    mlflow.log_metric("eval_loss", metrics["eval_loss"])
    mlflow.log_metric("base_eval_loss", metrics["base_eval_loss"])
    
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=NpzModel(),
        artifacts=artifacts,
        signature=signature
    )

print(f"Run ID: {run_id}")

# Register the model in Unity Catalog
model_uri = f"runs:/{run_id}/model"
registered = mlflow.register_model(
    model_uri=model_uri,
    name="workspace.mlpab4886e6.ftmodel0b3133",
    tags={
        "eval_loss": str(metrics["eval_loss"]),
        "base_eval_loss": str(metrics["base_eval_loss"])
    }
)
print(f"Registered model version: {registered.version}")
print(f"Model name: {registered.name}")